# Análise Exploratória Avançada - Datathon Passos Mágicos

## Introdução
Este notebook apresenta uma análise aprofundada dos indicadores educacionais da ONG **Passos Mágicos**. O objetivo é ir além da descrição estatística básica, buscando compreender as dinâmicas de desenvolvimento dos alunos, identificar padrões de risco e avaliar a eficácia do engajamento no desempenho acadêmico.

A análise está estruturada em:
1. **Preparação de Dados**: Limpeza e padronização.
2. **Análise Exploratória**: Distribuições e correlações fundamentais.
3. **Análise Avançada**: Relações multivariadas, grupos de performance e gap de percepção.
4. **Insights Estratégicos**: Conclusões para tomada de decisão pedagógica.

## 1. Preparação de Dados

Nesta etapa, carregamos os dados brutos e realizamos a padronização dos indicadores para garantir a consistência da análise.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set(style="whitegrid", palette="muted")

# Carregamento
df = pd.read_csv('../data/raw/PEDE_PASSOS_DATASET_FIAP.csv', sep=';')

# Seleção e Padronização
indicadores_originais = ['INDE_2020', 'IDA_2020', 'IEG_2020', 'IAA_2020', 'IPS_2020', 'IPP_2020', 'IAN_2020', 'IPV_2020']
df_eda = df[indicadores_originais].copy()
df_eda.columns = [col.split('_')[0] for col in df_eda.columns]

# Conversão Numérica
def to_numeric(col):
    return pd.to_numeric(col.astype(str).str.replace(',', '.'), errors='coerce')

for col in df_eda.columns:
    df_eda[col] = to_numeric(df_eda[col])

# Limpeza de Nulos no Target Principal (INDE)
df_eda = df_eda.dropna(subset=['INDE'])

print(f"Dataset processado com {df_eda.shape[0]} registros.")
df_eda.describe()

## 2. Análise Exploratória Fundamental

Exploração das distribuições e correlações para entender a base do desenvolvimento estudantil.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 18))
axes = axes.flatten()

for i, col in enumerate(df_eda.columns):
    sns.histplot(df_eda[col], kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(f'Distribuição de {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()

### Insights de Distribuição
- **INDE**: Apresenta uma distribuição multimodal, sugerindo a existência de diferentes perfis de alunos dentro da instituição.
- **IAN**: A alta concentração em valores elevados indica que a maioria dos alunos está em níveis adequados, mas a cauda inferior revela um grupo crítico que necessita de intervenção imediata para correção de fluxo escolar.

In [ ]:
plt.figure(figsize=(10, 7))
corr = df_eda.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Matriz de Correlação (Triangular)', fontsize=14, fontweight='bold')
plt.show()

### Insights de Correlação
- A correlação entre **IEG (Engajamento)** e **IDA (Aprendizagem)** é uma das mais fortes, reforçando que o envolvimento emocional e comportamental é o motor do sucesso acadêmico.
- O **IPV (Ponto de Virada)** mostra correlação moderada com o INDE, indicando que a percepção de transformação pessoal está alinhada ao crescimento global do aluno.

## 3. Análise Avançada

Nesta seção, aplicamos técnicas mais sofisticadas para identificar padrões ocultos e diferenciar perfis de performance.

### 3.1 Relações Multivariadas (Pairplot)

Analisando como múltiplos indicadores interagem simultaneamente.

In [ ]:
# Selecionando indicadores chave para o pairplot
cols_pair = ['INDE', 'IDA', 'IEG', 'IPS', 'IPV']
sns.pairplot(df_eda[cols_pair], diag_kind='kde', plot_kws={'alpha': 0.4, 's': 30, 'edgecolor': 'k'}, height=2.5)
plt.suptitle('Relações entre Indicadores Chave', y=1.02, fontsize=16, fontweight='bold')
plt.show()

**Impacto Educacional**: O pairplot revela que a relação entre IDA e IEG não é apenas linear, mas apresenta uma densidade maior em níveis elevados, sugerindo um 'efeito de aceleração': quanto mais o aluno se engaja, mais exponencial se torna seu ganho de aprendizagem.

### 3.2 Diferenciação de Performance (Top vs Bottom 20%)

O que realmente diferencia os alunos de alto desempenho dos que estão em risco?

In [ ]:
# Definindo os grupos baseados no INDE
q_low = df_eda['INDE'].quantile(0.20)
q_high = df_eda['INDE'].quantile(0.80)

top_20 = df_eda[df_eda['INDE'] >= q_high]
bottom_20 = df_eda[df_eda['INDE'] <= q_low]

# Calculando médias
comparison = pd.DataFrame({
    'Top 20% (High Performance)': top_20.mean(),
    'Bottom 20% (At Risk)': bottom_20.mean()
}).drop('INDE') # Removemos o INDE pois ele foi o critério de separação

# Visualização
comparison.plot(kind='barh', figsize=(12, 8), color=['#2ecc71', '#e74c3c'])
plt.title('Comparação de Médias: Alunos de Alta Performance vs Alunos em Risco', fontsize=14, fontweight='bold')
plt.xlabel('Valor Médio do Indicador')
plt.legend(loc='lower right')
plt.show()

**Padrões de Risco**: A maior disparidade entre os grupos está no **IDA (Aprendizagem)** e no **IEG (Engajamento)**. No entanto, note que o **IAN (Adequação de Nível)** também é significativamente menor no grupo em risco, sugerindo que o atraso escolar é um preditor fortíssimo de baixo desempenho global.

### 3.3 Análise do Gap de Percepção

Avaliando o descompasso entre a autoavaliação do aluno (IAA) e seu desempenho real (IDA).

In [ ]:
# Criando a variável Perception Gap
df_eda['perception_gap'] = df_eda['IAA'] - df_eda['IDA']

plt.figure(figsize=(14, 6))

# Distribuição do Gap
plt.subplot(1, 2, 1)
sns.histplot(df_eda['perception_gap'], kde=True, color='purple')
plt.axvline(0, color='red', linestyle='--')
plt.title('Distribuição do Gap de Percepção (IAA - IDA)', fontsize=12)
plt.xlabel('Gap (Positivo = Superestima / Negativo = Subestima)')

# Gap vs INDE
plt.subplot(1, 2, 2)
sns.scatterplot(data=df_eda, x='perception_gap', y='INDE', alpha=0.5, color='purple')
plt.axvline(0, color='red', linestyle='--')
plt.title('Gap de Percepção vs Performance Global (INDE)', fontsize=12)

plt.tight_layout()
plt.show()

**Insight Psico-Pedagógico**: 
- Alunos com **gap positivo elevado** (superestimam sua performance) tendem a ter um INDE menor. Isso pode indicar uma falta de consciência sobre suas lacunas de aprendizado, dificultando a busca por ajuda.
- Alunos com **gap próximo de zero** ou levemente negativo (mais autocríticos) estão concentrados nas faixas mais altas de INDE, sugerindo que a autoconsciência é uma característica de alunos de alto desempenho.

## 4. Conclusões e Insights Estratégicos

Com base na análise avançada, destacamos os seguintes pontos para a ONG Passos Mágicos:

1. **Foco no Engajamento**: O IEG é o principal catalisador do IDA. Programas que aumentem o vínculo do aluno com a instituição terão impacto direto nas notas.
2. **Alerta de Atraso Escolar**: O IAN baixo é um marcador claro de risco. Alunos com distorção idade-série precisam de trilhas de recuperação acelerada.
3. **Trabalho de Autoconsciência**: O 'Gap de Percepção' mostrou que alunos em risco muitas vezes não percebem suas dificuldades. Mentorias focadas em autoavaliação realista podem ser uma ferramenta poderosa.
4. **Ponto de Virada**: O IPV deve ser monitorado como um indicador antecedente de sucesso; alunos que começam a subir no IPV tendem a melhorar o INDE nos ciclos seguintes.